In [ ]:
### Prepare observed and discrepancy yield data and reload the splits from code1
import numpy as np
import pandas as pd
import torch
from tqdm import tqdm, trange
import function
import os

### Create the output directory
os.makedirs("../results/work/2", exist_ok=True)

### Load yield, socioeconomic, and environmental source data
yield_data = pd.read_csv("../data/yield.csv", engine="python").values[:, 1:]
social_data = pd.read_csv("../data/social.csv", engine="python").values[:, 1:]
natural_dataset = pd.read_csv("../data/natural.csv", engine="python")
natural_data = function.natural_feature_builder(natural_dataset).values

### Convert inputs to tensors and reshape them to the required dimensions
yield_data = torch.from_numpy(yield_data.astype("float")).float()
social_data = torch.from_numpy(social_data.astype("float")).float().reshape(-1, 22, 9)
natural_data = torch.from_numpy(natural_data.astype("float")).float().reshape(22, 12, -1, 8)[:, :9].permute(2, 0, 1, 3).reshape(654, 22, -1)

### Apply the shared completeness rule and construct sample-level arrays
county_index = np.arange(654).reshape(-1, 1)
yield_data, social_data, natural_data, county_index = function.filter_(yield_data, social_data, natural_data, county_index)
yield_usls, social_data, natural_data, county_index = function.flatten(yield_data, social_data, natural_data, county_index)
yield_obsvd, yield_trend, yield_resid = function.resolve(yield_data)

### Reload the saved test and cross-validation indices
data_index = pd.read_csv("../results/work/1/index.csv", engine="python")
test_index, n_splits, kf_indices = data_index.iloc[:, 0].dropna().to_numpy(dtype=np.int64, copy=True), 5, []
for i in range(0, 10, 2):
    kf_indices.append((data_index.iloc[:, i + 1].dropna().to_numpy(dtype=np.int64, copy=True), data_index.iloc[:, i + 2].dropna().to_numpy(dtype=np.int64, copy=True)))

In [ ]:
### Compute and export Integrated Gradients for the full-period experiment
### Define model configurations and output columns
model_iter = ["dnn", "cnn", "rnn", "lst", "gru", "att"]
num_instance_identity, require_explanation, batch_size, learning_rate, max_epoch, patience, train_breaker = 128, True, 32, 0.0003, 1000, 20, False
social_columns, natural_columns = ["PopD", "HeaR", "PuPr", "TePr", "FiRv", "FiEx", "ReSv", "InLn"], ["Radn", "AirT", "SoTe", "AirH", "SoMo", "Prec", "SuPr", "WiSp"]
feat_columns = ["county", "yield"] + social_columns
for i in range(9):
    for j in natural_columns:
        exec(f"feat_columns.append('{j}{i+1}')")
        
### Analyze yield observations and discrepancy yield separately
for i in ["obsvd", "resid"]:
    ### Train and evaluate each model across the five folds
    for model in tqdm(model_iter):
        for j in trange(n_splits):
            train_index, valid_index = kf_indices[j][0], kf_indices[j][1]
            exec(f"yield_train, yield_valid, yield_test = yield_{i}[train_index], yield_{i}[valid_index], yield_{i}[test_index]")
            social_train, social_valid, social_test = social_data[train_index], social_data[valid_index], social_data[test_index]
            natural_train, natural_valid, natural_test = natural_data[train_index], natural_data[valid_index], natural_data[test_index]
            yield_train_scaled, yield_valid_scaled, yield_test_scaled = function.scaler(yield_train, yield_valid, yield_test)
            social_train_scaled, social_valid_scaled, social_test_scaled = function.scaler(social_train, social_valid, social_test)
            natural_train_scaled, natural_valid_scaled, natural_test_scaled = function.scaler(natural_train, natural_valid, natural_test)
            output_kf = function.predictor(model, "addition", num_instance_identity, require_explanation,
                                           yield_train_scaled, social_train_scaled, natural_train_scaled,
                                           yield_valid_scaled, social_valid_scaled, natural_valid_scaled,
                                           batch_size, learning_rate, max_epoch, patience, train_breaker,
                                           yield_test_scaled, social_test_scaled, natural_test_scaled)
            feat_ipts = np.concatenate([county_index[test_index, 0].reshape(-1, 1), yield_test, output_kf[4]], 1)
            exec(f"feat_{model}_kf{j+1}_csv = pd.DataFrame(feat_ipts, columns=feat_columns)")

    ### Aggregate results across folds
    for j in model_iter:
        exec(f"feat_{j} = []")
    for j in range(n_splits):
        for k in model_iter:
            exec(f"feat_{k}.append(feat_{k}_kf{j+1}_csv.values)")
    for j in model_iter:
        exec(f"feat = np.concatenate(feat_{j})")
        exec(f"feat_{j}_csv = pd.DataFrame(feat[np.lexsort((feat[:,1],feat[:,0]))], columns=feat_columns)")
    index = [pd.DataFrame(test_index.reshape(-1, 1))]
    for j in range(n_splits):
        index.append(pd.DataFrame(kf_indices[j][0]))
        index.append(pd.DataFrame(kf_indices[j][1]))
    index_csv = pd.concat(index, axis=1)
    index_csv.columns = ["test", "kf1_t", "kf1_v", "kf2_t", "kf2_v", "kf3_t", "kf3_v", "kf4_t", "kf4_v", "kf5_t", "kf5_v"]

    ### Export the processed results
    index_csv.to_csv("../results/work/2/index.csv", index=False)
    for j in model_iter:
        exec(f"feat_{j}_csv.to_csv('../results/work/2/{i}_{j}.csv', index=False)")
        for k in range(n_splits):
            exec(f"feat_{j}_kf{k+1}_csv.to_csv('../results/work/2/{i}_{j}_kf{k+1}.csv', index=False)")